In [1]:
!pip install "pymilvus[model]==2.5.10" openai==1.82.0 requests==2.32.3 tqdm==4.67.1 torch==2.7.0

Looking in indexes: https://repo.huaweicloud.com/repository/pypi/simple


In [12]:
import os

# 从环境变量获取 DeepSeek API Key
api_key = os.getenv("DEEPSEEK_API_KEY")

In [13]:
api_key

'sk-d6207b86cbd346959abc91b16a530878'

In [14]:
from glob import glob

text_lines = []

for file_path in glob("code/deepseek-quickstart/deepseek/api/*.md", recursive=True):
    print(file_path)
    with open(file_path, "r") as file:
        file_text = file.read()

    text_lines += file_text.split("# ")

code/deepseek-quickstart/deepseek/api/mfd.md


In [15]:
len(text_lines)

30

In [16]:
from openai import OpenAI

deepseek_client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 的基地址
)

In [17]:

from pymilvus import model as milvus_model

# OpenAI国内代理 https://api.apiyi.com/token 
embedding_model = milvus_model.dense.OpenAIEmbeddingFunction(
    model_name='text-embedding-3-large', # Specify the model name
    api_key='sk-Yma7bjwcDvGyiSgWB8074bD82332461e90EeE65dF5510f4e', # Provide your OpenAI API key
    base_url='https://api.apiyi.com/v1',
    dimensions=512
)

/home/huaj/miniconda3/envs/deepseek/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
test_embedding = embedding_model.encode_queries(["This is a test"])[0]
embedding_dim = len(test_embedding)
print(embedding_dim)
print(test_embedding[:10])

512
[-0.02814663  0.00428726 -0.01852599  0.08190062 -0.03156214 -0.05275258
 -0.04885425  0.12481797 -0.0208328   0.03966279]


In [23]:
test_embedding_0 = embedding_model.encode_queries(["That is a test"])[0]
print(test_embedding_0[:10])

[-0.00578664  0.02242682 -0.01892621  0.12811586 -0.01249751 -0.07321841
 -0.00281971  0.08617394 -0.04377401  0.03073668]


In [19]:
from pymilvus import MilvusClient

milvus_client = MilvusClient(uri="./milvus_demo2.db")

collection_name = "my_rag_collection"

In [20]:
if milvus_client.has_collection(collection_name):
    milvus_client.drop_collection(collection_name)

In [24]:
milvus_client.create_collection(
    collection_name=collection_name,
    dimension=embedding_dim,
    metric_type="IP",  # 内积距离
    consistency_level="Strong",  # 支持的值为 (`"Strong"`, `"Session"`, `"Bounded"`, `"Eventually"`)。更多详情请参见 https://milvus.io/docs/consistency.md#Consistency-Level。
)

In [26]:
from tqdm import tqdm

data = []

doc_embeddings = embedding_model.encode_documents(text_lines)

for i, line in enumerate(tqdm(text_lines, desc="Creating embeddings")):
    data.append({"id": i, "vector": doc_embeddings[i], "text": line})

milvus_client.insert(collection_name=collection_name, data=data)

Creating embeddings: 100%|██████████| 30/30 [00:00<00:00, 449389.71it/s]


{'insert_count': 30, 'ids': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29], 'cost': 0}

In [27]:
question = "请介绍登记资料，哪些人可以查看登记资料？?"

In [28]:
search_res = milvus_client.search(
    collection_name=collection_name,
    data=embedding_model.encode_queries(
        [question]
    ),  # 将问题转换为嵌入向量
    limit=3,  # 返回前3个结果
    search_params={"metric_type": "IP", "params": {}},  # 内积距离
    output_fields=["text"],  # 返回 text 字段
)

In [29]:
import json

retrieved_lines_with_distances = [
    (res["entity"]["text"], res["distance"]) for res in search_res[0]
]
print(json.dumps(retrieved_lines_with_distances, indent=4))

[
    [
        "\u7b2c\u4e00\u7ae0 \u4e00\u822c\u89c4\u5b9a\n\n**\u7b2c\u4e8c\u767e\u96f6\u56db\u6761** \u4e3a\u4e86\u660e\u786e\u7269\u7684\u5f52\u5c5e\uff0c\u5145\u5206\u53d1\u6325\u7269\u7684\u6548\u7528\uff0c\u4fdd\u62a4\u6743\u5229\u4eba\u7684\u5408\u6cd5\u6743\u76ca\uff0c\u7ef4\u62a4\u793e\u4f1a\u7ecf\u6d4e\u79e9\u5e8f\uff0c\u5236\u5b9a\u672c\u7f16\u3002\n\n**\u7b2c\u4e8c\u767e\u96f6\u4e94\u6761** \u672c\u7f16\u8c03\u6574\u56e0\u7269\u7684\u5f52\u5c5e\u548c\u5229\u7528\u4ea7\u751f\u7684\u6c11\u4e8b\u5173\u7cfb\u3002\n\n**\u7b2c\u4e8c\u767e\u96f6\u516d\u6761** \u56fd\u5bb6\u575a\u6301\u548c\u5b8c\u5584\u793e\u4f1a\u4e3b\u4e49\u516c\u6709\u5236\u4e3a\u4e3b\u4f53\u3001\u591a\u79cd\u6240\u6709\u5236\u7ecf\u6d4e\u5171\u540c\u53d1\u5c55\u7684\u57fa\u672c\u7ecf\u6d4e\u5236\u5ea6\u3002\n\u56fd\u5bb6\u5de9\u56fa\u548c\u53d1\u5c55\u516c\u6709\u5236\u7ecf\u6d4e\uff0c\u9f13\u52b1\u3001\u652f\u6301\u548c\u5f15\u5bfc\u975e\u516c\u6709\u5236\u7ecf\u6d4e\u7684\u53d1\u5c55\u3002\n\u56fd\u5bb6\u5

In [30]:
context = "\n".join(
    [line_with_distance[0] for line_with_distance in retrieved_lines_with_distances]
)

In [31]:
context

'第一章 一般规定\n\n**第二百零四条** 为了明确物的归属，充分发挥物的效用，保护权利人的合法权益，维护社会经济秩序，制定本编。\n\n**第二百零五条** 本编调整因物的归属和利用产生的民事关系。\n\n**第二百零六条** 国家坚持和完善社会主义公有制为主体、多种所有制经济共同发展的基本经济制度。\n国家巩固和发展公有制经济，鼓励、支持和引导非公有制经济的发展。\n国家实行社会主义市场经济，保障一切市场主体的平等法律地位和发展权利。\n\n**第二百零七条** 国家、集体、私人的物权和其他权利人的物权受法律平等保护，任何组织或者个人不得侵犯。\n\n**第二百零八条** 不动产权利的设立、变更、转让和消灭，应当依照法律规定登记。动产物权的设立和转让，应当依照法律规定交付。\n\n**第二百零九条** 不动产物权的设立、变更、转让和消灭，经依法登记，发生效力；未经登记，不发生效力，但是法律另有规定的除外。\n依法属于国家所有的自然资源，所有权可以不登记。\n\n**第二百一十条** 不动产登记，由不动产所在地的登记机构办理。\n国家对不动产实行统一登记制度。统一登记的范围、登记机构和登记办法，由法律、行政法规规定。\n\n**第二百一十一条** 当事人申请登记，应当根据不同登记事项提供材料。\n申请登记材料以及登记事项相关信息，可以公开查询。\n\n**第二百一十二条** 登记机构应当履行下列职责：\n（一）审查申请人提供的材料；\n（二）询问申请人；\n（三）如实、及时登记；\n（四）法律、行政法规规定的其他职责。\n申请登记的不动产存在尚未解决的权属争议的，登记机构应当不予登记，并书面告知申请人。\n\n**第二百一十三条** 登记机构不得有下列行为：\n（一）要求对不动产进行评估；\n（二）以不动产登记为条件收取其他费用；\n（三）超出登记职责范围的其他行为。\n\n**第二百一十四条** 不动产物权的设立、变更、转让和消灭，依照法律规定应当登记的，自记载于不动产登记簿时发生效力。\n\n**第二百一十五条** 不动产登记簿由登记机构管理。\n不动产登记簿应当采用纸质形式或者电子形式。\n不动产登记簿采用电子形式的，应当备份。\n\n**第二百一十六条** 不动产登记簿是物权归属和内容的根据。\n不动产登记簿记载的事项与不动产权属证书记载的事项不一致的

In [32]:
SYSTEM_PROMPT = """
Human: 你是一个 AI 助手。你能够从提供的上下文段落片段中找到问题的答案。
"""
USER_PROMPT = f"""
请使用以下用 <context> 标签括起来的信息片段来回答用 <question> 标签括起来的问题。
<context>
{context}
</context>
<question>
{question}
</question>
"""

In [33]:
response = deepseek_client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ],
)
print(response.choices[0].message.content)

根据提供的上下文，关于登记资料（特指不动产登记资料）的查询，以下人员可以查看：

1. **权利人**：即不动产登记簿上记载的权利人。
2. **利害关系人**：指与不动产登记事项有法律上利害关系的人。

具体依据如下：
- **第二百一十八条**规定：“权利人、利害关系人可以申请查询、复制不动产登记资料，登记机构应当提供。”
- **第二百一十九条**进一步明确：“利害关系人可以申请查询不动产登记资料。申请查询的，登记机构应当提供。”

此外，**第二百一十一条**还提到：“申请登记材料以及登记事项相关信息，可以公开查询。” 这表明登记资料在原则上具有一定的公开性，但具体查询主体仍以权利人和利害关系人为主。
